# Tutorial 1.8: Prompt Optimization with GEPA

![](images/9_Prompt-Optimization-with-GEPA.png)

## Automatically Improve Prompts Using MLflow's GEPA Integration

In Tutorial 1.5, we manually iterated on prompts — writing better versions by hand and versioning them in the Prompt Registry. But what if an algorithm could do this automatically?

This notebook demonstrates **GEPA (Genetic-Pareto)**, an automatic prompt optimization algorithm integrated into MLflow via `mlflow.genai.optimize_prompts()`.

### What You'll Learn

- How GEPA automatically improves prompts
- Using `mlflow.genai.optimize_prompts()` with the Prompt Registry
- Writing a custom `@scorer` for fast, exact-match evaluation
- Comparing original vs. optimized prompts

### Prerequisites
- Completed Notebook 1.5 (Prompt Management) and 1.7 (Evaluating Agents)
- Understanding of the Prompt Registry and evaluation scorers

### Estimated Time: 10-15 minutes

---
## Step 1: How GEPA Works

**GEPA (Genetic-Pareto)** optimizes prompts through an iterative cycle:

```
1. EVALUATE  →  Run the prompt on training examples, score with a judge
2. REFLECT   →  Use an LLM to analyze failures and propose improvements
3. MUTATE    →  Generate improved prompt variations
4. SELECT    →  Keep the best-performing candidates (Pareto-optimal)
5. REPEAT    →  Continue until budget exhausted or convergence
```

### Manual vs. Automatic Optimization

| Approach | Method | Effort | Consistency |
|----------|--------|--------|-------------|
| **Manual** (Notebook 1.5) | Human writes better prompts | High | Variable |
| **GEPA** (This notebook) | Algorithm evolves prompts | Low | Systematic |

### Integration with Prompt Registry

GEPA works directly with MLflow's Prompt Registry:
- **Reads** your registered prompt as the starting point
- **Optimizes** it through the evaluate-reflect-mutate cycle
- **Registers** the improved version automatically as a new version

> **Note:** GEPA requires the `gepa` package. Install it with: `pip install gepa`

---
## Step 2: Environment Setup

In [1]:
import mlflow
from dotenv import load_dotenv
from utils.clnt_utils import is_databricks_ai_gateway_client, get_databricks_ai_gateway_client, get_openai_client, get_ai_gateway_model_names

# Load environment
load_dotenv()

# Configure MLflow
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("08-prompt-optimization")

# Configure client and model based on provider
use_databricks_provider = is_databricks_ai_gateway_client()
if use_databricks_provider:
    client = get_databricks_ai_gateway_client()
    model_name = get_ai_gateway_model_names()[0]
    optimizer_model = f"databricks:/{model_name}"
else:
    client = get_openai_client()
    model_name = "gpt-5-mini"
    optimizer_model = f"openai:/{model_name}"

# Enable autologging
mlflow.openai.autolog()

print("✅ Environment configured")
print(f"   Provider: {'Databricks AI Gateway' if use_databricks_provider else 'OpenAI'}")
print(f"   Model: {model_name}")
print(f"   Optimizer model: {optimizer_model}")
print(f"   Tracking URI: {mlflow.get_tracking_uri()}")

2026/05/04 18:12:28 INFO mlflow.tracking.fluent: Experiment with name '08-prompt-optimization' does not exist. Creating a new experiment.


✅ Environment configured
   Provider: OpenAI
   Model: gpt-5-mini
   Optimizer model: openai:/gpt-5-mini
   Tracking URI: http://localhost:5000


---
## Step 3: Register a Baseline Prompt

We'll use the same basic Q&A prompt from Notebook 1.5's Prompt Library (`qa_simple`). We register it fresh here so this notebook is self-contained.

This minimal prompt is an ideal optimization target — it has maximum room for GEPA to improve it.

In [2]:
# Register the baseline prompt (same template as qa_simple from Notebook 1.5)
baseline_prompt = mlflow.genai.register_prompt(
    name="gepa-qa-simple",
    template="Answer this question: {{ question }}",
    commit_message="Baseline prompt for GEPA optimization",
    tags={"author": "jules", "use_case": "Simple Q&A", "status": "baseline"}
)

print("✅ Baseline prompt registered")
print(f"   Name: {baseline_prompt.name}")
print(f"   Version: {baseline_prompt.version}")
print(f"   URI: {baseline_prompt.uri}")
print(f"   Template: '{baseline_prompt.template}'")

2026/05/04 18:12:29 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: gepa-qa-simple, version 1


✅ Baseline prompt registered
   Name: gepa-qa-simple
   Version: 1
   URI: prompts:/gepa-qa-simple/1
   Template: 'Answer this question: {{ question }}'


---
## Step 4: Prepare Training Data, Scorer, and Predict Function

GEPA needs three things:
1. **Training data** — example input/output pairs so it can evaluate prompt quality
2. **Scorer** — a function that scores how well the output matches expectations
3. **Predict function** — a callable that loads the prompt, fills it, and calls the LLM

### Design Strategy: Short-Answer Exact Match

We use **short-answer factual questions** with a **custom exact-match scorer** (same pattern
as the [HotpotQA optimization benchmark](https://mlflow.org/docs/latest/llms/prompt-optimization)).

**Why this works:** The bare-bones prompt produces verbose responses like
*"The capital of Japan is Tokyo, which is located on..."* — but the expected answer is just `"Tokyo"`.
Exact match fails on every verbose answer, giving GEPA a near-zero baseline. GEPA then learns
to add conciseness instructions ("respond with only the answer"), and the score jumps dramatically.

**Why it's fast:** The custom scorer is pure Python string comparison — no LLM judge calls needed.
Each evaluation pass only costs LLM calls for the predict function, not for scoring.

In [3]:
from mlflow.genai import optimize_prompts, scorer
from mlflow.genai.optimize.optimizers import GepaPromptOptimizer
from mlflow.genai.judges import CategoricalRating
from mlflow.entities import Feedback
from gepa.utils import NoImprovementStopper

# Custom exact-match scorer — pure Python, no LLM judge calls needed.
# Returns YES (1.0) if normalized output matches expected answer, NO (0.0) otherwise.
@scorer
def exact_match(outputs: str, expectations: dict) -> Feedback:
    expected = expectations["expected_response"].strip().lower()
    actual = outputs.strip().lower().rstrip(".")
    return Feedback(
        name="exact_match",
        value=CategoricalRating.YES if actual == expected else CategoricalRating.NO,
    )

# Training data: short-answer factual questions with unambiguous 1-3 word answers.
# The bare-bones prompt will produce verbose full-sentence answers that fail exact match.
# GEPA must learn to add conciseness instructions to pass.
train_data = [
    {"inputs": {"question": "What is the chemical symbol for gold?"}, "expectations": {"expected_response": "Au"}},
    {"inputs": {"question": "What planet is closest to the sun?"}, "expectations": {"expected_response": "Mercury"}},
    {"inputs": {"question": "In what year did the Titanic sink?"}, "expectations": {"expected_response": "1912"}},
    {"inputs": {"question": "What is the capital of Japan?"}, "expectations": {"expected_response": "Tokyo"}},
    {"inputs": {"question": "Who wrote Romeo and Juliet?"}, "expectations": {"expected_response": "William Shakespeare"}},
    {"inputs": {"question": "What is the largest planet in our solar system?"}, "expectations": {"expected_response": "Jupiter"}},
    {"inputs": {"question": "What element does the symbol O represent on the periodic table?"}, "expectations": {"expected_response": "Oxygen"}},
    {"inputs": {"question": "What continent is Egypt in?"}, "expectations": {"expected_response": "Africa"}},
    {"inputs": {"question": "What is the hardest natural substance on Earth?"}, "expectations": {"expected_response": "Diamond"}},
    {"inputs": {"question": "What is the largest ocean on Earth?"}, "expectations": {"expected_response": "Pacific Ocean"}},
]


# Predict function: GEPA calls this repeatedly during optimization.
# During optimization, GEPA patches PromptVersion.template so that
# load_prompt() returns the MUTATED template instead of the original.
def predict_qa(question: str) -> str:
    """Load the prompt from the registry, fill it, and call the LLM."""
    prompt = mlflow.genai.load_prompt(baseline_prompt.uri)
    filled = prompt.format(question=question)

    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": filled}],
    )
    return response.choices[0].message.content


print(f"✅ Training data prepared: {len(train_data)} short-answer examples")
print("✅ Custom exact_match scorer defined (no LLM judge calls)")
print("✅ Predict function defined")
print(f"   Loads prompt from: {baseline_prompt.uri}")

✅ Training data prepared: 10 short-answer examples
✅ Custom exact_match scorer defined (no LLM judge calls)
✅ Predict function defined
   Loads prompt from: prompts:/gepa-qa-simple/1


---
## Step 5: Run GEPA Optimization

Now we run the optimization. GEPA will:
1. Evaluate the baseline prompt using the custom `exact_match` scorer
2. Reflect on failures (verbose answers that don't match the short expected answers)
3. Generate improved prompt variations with conciseness instructions
4. Register the optimized prompt as a new version in the Prompt Registry

We also configure **early stopping** via `NoImprovementStopper` — if GEPA sees no score
improvement for 3 consecutive iterations, it stops early rather than burning the remaining budget.

> **Note:** The short-answer exact-match pattern is adapted from MLflow's
> [HotpotQA prompt optimization example](https://mlflow.org/docs/latest/llms/prompt-optimization),
> which uses the same approach — factual questions with terse expected answers scored via strict
> string matching — to demonstrate dramatic GEPA improvements on a Q&A task.

> **Note:** This typically takes 5-10 minutes. The custom scorer runs instantly (no LLM judge),
> so the only LLM calls are from the predict function — 10 per evaluation pass.

In [4]:
import logging
from ipykernel.iostream import OutStream

# === Fix for GEPA Unicode surrogate characters ===
# GEPA's internal output contains Unicode surrogates that crash Jupyter's
# ZMQ/tornado JSON encoder. We fix at three levels:

def _sanitize_surrogates(obj):
    """Recursively replace Unicode surrogates in an object tree."""
    if isinstance(obj, str):
        return obj.encode("utf-8", errors="replace").decode("utf-8")
    elif isinstance(obj, bytes):
        return obj
    elif isinstance(obj, dict):
        return {_sanitize_surrogates(k): _sanitize_surrogates(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return type(obj)(_sanitize_surrogates(item) for item in obj)
    return obj

# Level 1: Patch OutStream.write at the CLASS level to sanitize all output.
_orig_outstream_write = OutStream.write

def _safe_outstream_write(self, string):
    if isinstance(string, str):
        string = string.encode("utf-8", errors="replace").decode("utf-8")
    return _orig_outstream_write(self, string)

OutStream.write = _safe_outstream_write

# Level 2: Patch the kernel session's pack function to handle surrogates.
try:
    _kernel = get_ipython().kernel
    _orig_pack = _kernel.session.pack

    def _safe_pack(obj):
        try:
            return _orig_pack(obj)
        except (UnicodeEncodeError, TypeError):
            return _orig_pack(_sanitize_surrogates(obj))

    _kernel.session.pack = _safe_pack
except Exception:
    pass

# Level 3: Suppress async/tornado error log noise.
for _logger_name in ("tornado.general", "tornado.application", "asyncio"):
    logging.getLogger(_logger_name).setLevel(logging.CRITICAL)

# Run GEPA prompt optimization
print("🔄 Running GEPA prompt optimization...\n")
print("   This will iterate through evaluate → reflect → mutate → select cycles.")
print("   Budget: 100 metric calls | Early stop: 3 iterations without improvement")
print("   Scorer: exact_match (no LLM judge — fast!)\n")

result = optimize_prompts(
    predict_fn=predict_qa,
    train_data=train_data,
    prompt_uris=[baseline_prompt.uri],
    optimizer=GepaPromptOptimizer(
        reflection_model=optimizer_model,
        max_metric_calls=100,
        display_progress_bar=False,
        gepa_kwargs={
            "stop_callbacks": NoImprovementStopper(max_iterations_without_improvement=3),
        },
    ),
    scorers=[exact_match],
)

print("\n✅ GEPA optimization complete!")

2026/05/04 18:12:29 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/05/04 18:12:29 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


🔄 Running GEPA prompt optimization...

   This will iterate through evaluate → reflect → mutate → select cycles.
   Budget: 100 metric calls | Early stop: 3 iterations without improvement
   Scorer: exact_match (no LLM judge — fast!)



/Users/jules/git-repos/mlflow-genai-tutorials/.venv/lib/python3.12/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


Iteration 0: Base program full valset score: 0.1 over 10 / 10 examples
Iteration 1: Selected program 0 score: 0.1
Iteration 1: Proposed new text for gepa-qa-simple: Task summary
- Input: a single prompt of the form "Answer this question: {{ question }}" (the assistant will receive the full string; the question text follows the colon).
- Output: a single, concise canonical answer to the question — nothing else.

Primary instruction (behavior)
1. Extract the question text after "Answer this question:" and determine the single most direct, widely accepted answer (usually a single word or short noun phrase / proper name).
2. Return only that answer, and only that answer:
   - No leading/trailing commentary, no preceding articles ("The", "It is"), no explanations or qualifiers, no additional sentences, no bullets or lists unless the question explicitly requires multiple items.
   - No trailing punctuation (do not end with a period).
   - Use conventional capitalization and conventional word

2026/05/04 18:13:36 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: gepa-qa-simple, version 2


Iteration 4: All subsample scores perfect. Skipping.
Iteration 4: Reflective mutation did not propose a new candidate
🏃 View run youthful-squid-818 at: http://localhost:5000/#/experiments/1/runs/432268a1d8b842df8502eb993d2ad0c5
🧪 View experiment at: http://localhost:5000/#/experiments/1

✅ GEPA optimization complete!


Trace(trace_id=tr-df525efec1e5f1a1b3eb40d8448ea9d4)

---
## Step 6: Compare Original vs. Optimized Prompt

In [5]:
def _safe(s):
    """Strip Unicode surrogates for safe display in Jupyter."""
    if isinstance(s, str):
        return s.encode("utf-8", errors="replace").decode("utf-8")
    return str(s)

# Display before/after comparison
print("=" * 70)
print("📊 GEPA Optimization Results")
print("=" * 70)

print("\n📈 Score Improvement:")
if result.initial_eval_score is not None:
    print(f"   Initial score: {result.initial_eval_score:.3f}")
else:
    print("   Initial score: N/A")
if result.final_eval_score is not None:
    print(f"   Final score:   {result.final_eval_score:.3f}")
else:
    print("   Final score:   N/A")
if result.initial_eval_score is not None and result.final_eval_score is not None:
    improvement = result.final_eval_score - result.initial_eval_score
    print(f"   Improvement:   {improvement:+.3f}")

# Load the optimized prompt directly from the registry to ensure we
# see the actual registered version (not just the in-memory object)
optimized = result.optimized_prompts[0]
registry_prompt = mlflow.genai.load_prompt(f"prompts:/{optimized.name}/{optimized.version}")

print(f"\n📝 Original Prompt (version {baseline_prompt.version}):")
print(f"   '{baseline_prompt.template}'")

print(f"\n🚀 Optimized Prompt (version {optimized.version}):")
print(f"   '{_safe(registry_prompt.template)}'")

if baseline_prompt.template.strip() == _safe(registry_prompt.template).strip():
    print("\n⚠️  Note: The optimized template is identical to the baseline.")
    print("   This can happen when the baseline already scores well on the")
    print("   training data. Try adding harder examples or increasing the budget.")

print("\n🔗 The optimized prompt has been automatically registered")
print(f"   as version {optimized.version} in the Prompt Registry!")
print(f"   View it in MLflow UI → Prompt Registry → {_safe(optimized.name)}")

print("\n" + "=" * 70)
print("\n💡 Key Takeaway:")
print("   GEPA automatically learned to add structure, instructions,")
print("   and constraints that we would normally write by hand.")
print("   Combined with the Prompt Registry, optimized prompts are")
print("   versioned and ready for deployment via aliases.")

📊 GEPA Optimization Results

📈 Score Improvement:
   Initial score: 0.100
   Final score:   1.000
   Improvement:   +0.900

📝 Original Prompt (version 1):
   'Answer this question: {{ question }}'

🚀 Optimized Prompt (version 2):
   'Task summary
- Input: a single prompt of the form "Answer this question: {{ question }}" (the assistant will receive the full string; the question text follows the colon).
- Output: a single, concise canonical answer to the question — nothing else.

Primary instruction (behavior)
1. Extract the question text after "Answer this question:" and determine the single most direct, widely accepted answer (usually a single word or short noun phrase / proper name).
2. Return only that answer, and only that answer:
   - No leading/trailing commentary, no preceding articles ("The", "It is"), no explanations or qualifiers, no additional sentences, no bullets or lists unless the question explicitly requires multiple items.
   - No trailing punctuation (do not end with 

---
## Summary

In this notebook, you learned:

1. How **GEPA** automatically optimizes prompts through evaluate-reflect-mutate cycles
2. Using `mlflow.genai.optimize_prompts()` with the **Prompt Registry**
3. Writing a **custom `@scorer`** for fast exact-match evaluation (no LLM judge needed)
4. Using **`NoImprovementStopper`** for early stopping when GEPA converges
5. Comparing **before/after** prompt quality — from verbose failures to terse correct answers
6. Optimized prompts are **automatically versioned** in the registry

### Key Takeaways

- **Scorer design matters**: Exact-match scoring on short answers creates a clear gap between a bare prompt (verbose, fails) and an optimized prompt (terse, passes)
- **Custom scorers are fast**: Pure Python scoring eliminates LLM judge calls, cutting optimization time roughly in half
- **Early stopping saves time**: `NoImprovementStopper` prevents wasting budget once GEPA converges
- **Registry integration**: Optimized prompts flow directly into the Prompt Registry for versioning and deployment
- **Combine approaches**: Use GEPA for initial optimization, then fine-tune manually if needed

### What's Next?

**📓 Notebook 1.9: Complete RAG Application**

Learn how to:
- Build a full RAG pipeline with end-to-end tracing
- Evaluate RAG quality with RAGAS metrics
- Track performance, cost, and retrieval quality